# Scenario: Auditing the "Alert Fatigue" Factor

In [3]:
import pandas as pd
import sqlite3
# creating dataset of blood pressure readings
bp_data = {
    "reading_id": [801, 802, 803, 804, 805, 806, 807, 808],
    "patient_id": ["P-01", "P-02", "P-03", "P-04", "P-05", "P-06", "P-07", "P-08"],
    "systolic_bp": [135, 142, 155, 139, 141, 160, 148, 120]   
}
# adding dataset to DataFrame
df_data = pd.DataFrame(bp_data)
# connecting to the slqite
connt = sqlite3.connect(":memory:")
df_data.to_sql("vitals", connt, index = False, if_exists = "replace")
# creating a function to run query
def run_query(query):
    return pd.read_sql_query(query, connt)

print("************************** Day 14 Threshold Audit Database is ready!!! *************")
    

************************** Day 14 Threshold Audit Database is ready!!! *************


# The "Current State" Audit

In [8]:
# query for all data to review 
all_data = "SELECT * FROM vitals"
print("*************************** all data to review ********************")
display(run_query(all_data))
print()
# query query to count how many patients are currently being flagged by the AI (where systolic_bp > 140)
flagged_data = """ 
SELECT patient_id, systolic_bp FROM vitals
WHERE systolic_bp > 140
"""
print("*************************** flagged data ***************************")
display(run_query(flagged_data))

*************************** all data to review ********************


,reading_id,patient_id,systolic_bp
0,801,P-01,135
1,802,P-02,142
2,803,P-03,155
3,804,P-04,139
4,805,P-05,141
5,806,P-06,160
6,807,P-07,148
7,808,P-08,120



*************************** flagged data ***************************


,patient_id,systolic_bp
0,P-02,142
1,P-03,155
2,P-05,141
3,P-06,160
4,P-07,148


# The "What If" Scenario

In [13]:
# query that categorizes patients into three groups: 'Normal' (<140), 'Borderline' (140-150), and 'Critical' (>150).
whatif_data = """
SELECT
    CASE
        WHEN systolic_bp < 140 THEN "Normal"
        WHEN systolic_bp BETWEEN 140 AND 150 THEN "Borderline"
        ELSE "Critical"
    END AS pb_category,
    COUNT(*)  AS patient_count
FROM vitals
GROUP BY pb_category
"""
print("******************************* what if scenario *******************")
display(run_query(whatif_data))


******************************* what if scenario *******************


,pb_category,patient_count
0,Borderline,3
1,Critical,2
2,Normal,3
